# 03 — Starter curricula, verified labels and grouped splits

## Goal
Create a **small foundation dataset**, not a massive training corpus or a trained
AHR Go model. Six families cover graph coloring, Boolean clauses, alternating
quantified clauses, and bounded Go on 3×3, 5×5 and 9×9 boards.

**Inputs:** notebooks 00–02 and the AHR Go `.venv` kernel. No downloads or private data.
**Outputs:** immutable `data/starter_v1_<hash>/` JSONL shards and a manifest;
`reports/03_dataset_validation.json`; resumable `checkpoints/datasets-<version>/`.
**Expected runtime:** 1–5 minutes on CPU for the fixed small defaults; GPUs are not
needed. There is no optimizer or AHR training step.
**Resume:** restart the kernel and Run All. Latest valid checkpoints restore the
accepted groups, candidate cursor, deterministic seed scheme, configuration and
metrics. Completed reruns validate the same files without adding examples. A changed
configuration or exported definition creates a new version, never overwrites old data.

## Method and limits
Known labels must agree with a separate exhaustive reference; every SAT witness or
strategy is checked. `UNKNOWN` is quarantined, with a null training target. Go tasks
are finite-horizon **area-score proxies** or explicitly terminal-win tasks. These are
not a life-and-death problem collection and do not imply eventual perfect play.

Equivalent variable renamings and Go rotations/reflections are grouped before
splitting. For these tiny ordinary CSPs, canonical complete truth relations also
merge logically equivalent constraint presentations under variable renaming. QCSP
canonicalization preserves the quantifier prefix and its information order. Go
grouping includes the complete superko history; horizons/objectives from one state
remain in one group. No claim of all possible cross-domain semantic equivalence is made.

In [ ]:
from pathlib import Path
import os, json, hashlib, time, sys
import nbformat
PROJECT_ROOT = Path(os.environ.get("AHR_GO_PROJECT", Path.cwd())).resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "remote_notebooks/ahr_go/notebooks").is_dir(): PROJECT_ROOT = PROJECT_ROOT / "remote_notebooks/ahr_go"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
DEPENDENCY_NAMES = ["00_instance_setup_and_recovery.ipynb", "01_go_rules_and_independent_verifier.ipynb", "02_go_to_csp_and_qcsp.ipynb"]
DEPENDENCY_HASHES = {}
for name in DEPENDENCY_NAMES:
    notebook = nbformat.read(NOTEBOOK_DIR / name, as_version=4)
    definitions = [cell.source for cell in notebook.cells if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", [])]
    assert definitions, name
    DEPENDENCY_HASHES[name] = hashlib.sha256("\n\n".join(definitions).encode()).hexdigest()
    for source in definitions: exec(compile(source, name + ":definitions", "exec"), globals())
this_notebook = nbformat.read(NOTEBOOK_DIR / "03_curricula_datasets_and_splits.ipynb", as_version=4)
GENERATOR_HASH = hashlib.sha256("\n\n".join(cell.source for cell in this_notebook.cells if cell.cell_type == "code").encode()).hexdigest()
CONFIG = {"schema": "ahr_go_starter_v1", "seed": 20260923, "generator_code_sha256": GENERATOR_HASH,
          "export_sha256": DEPENDENCY_HASHES,
          "groups": {"graph_coloring": 16, "boolean_sat": 32, "quantified_sat": 32, "go_3x3": 16, "go_5x5": 12, "go_9x9": 8},
          "max_candidates_per_family": 2000, "max_nodes_per_problem": 100000,
          "go_horizons": {"3": 2, "5": 2, "9": 1}, "small_board_komi": 0.5, "nine_board_komi": 7.5,
          "checkpoint_candidates": 8, "checkpoint_seconds": 60, "session_seconds": 480,
          "split": "sorted known-group hash; approximately 80/10/10; at least one known dev/test group per family; UNKNOWN groups separately hash-assigned"}
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
VERSION = "starter_v1_" + CONFIG_HASH[:12]
DATA_DIR = PROJECT_ROOT / "data" / VERSION
RUN_DIR = PROJECT_ROOT / "checkpoints" / ("datasets-" + VERSION)
print(f"Project: {PROJECT_ROOT}\nVersion: {VERSION}\nRequested unique groups: {sum(CONFIG['groups'].values())}\nTraining: not started")

## 1. Deterministic finite problems and canonical groups
The seed for each candidate depends only on family, candidate index and master seed.
No live random generator must survive a restart. Every accepted base problem receives
a renamed or reflected version in the **same group**. Exact duplicate rows are removed.
The group key does not contain the label. Small-board komi is 0.5; 9×9 uses 7.5.

In [ ]:
from itertools import permutations, product, combinations
from dataclasses import asdict
from collections import Counter, defaultdict
import random, hashlib, json, time, copy

def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)

def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()

def candidate_rng(seed, family, index):
    return random.Random(int(digest_json([seed, family, index])[:16], 16))

def rename_csp(problem):
    names = list(problem.domains)
    mapping = {name: f"renamed_{len(names)-1-i}" for i, name in enumerate(names)}
    return CSP({mapping[name]: values for name, values in reversed(list(problem.domains.items()))},
               tuple(TableConstraint(tuple(mapping[name] for name in c.scope), c.allowed) for c in reversed(problem.constraints)),
               tuple((q, mapping[name]) for q, name in problem.prefix))

def canonical_csp(problem):
    validate_csp(problem)
    names = tuple(problem.domains)
    assert len(names) <= 5, "Exhaustive canonicalization is only for tiny starter tasks."
    # Complete satisfying relation, computed directly rather than using a search label.
    relation = []
    for values in product(*(problem.domains[name] for name in names)):
        assignment = dict(zip(names, values))
        if all(tuple(assignment[name] for name in c.scope) in c.allowed for c in problem.constraints):
            relation.append(values)
    forms = []
    for order in permutations(range(len(names))):
        renaming = {names[old]: new for new, old in enumerate(order)}
        forms.append(canonical_json({"domains": [list(problem.domains[names[i]]) for i in order],
                                     "relation": sorted([tuple(row[i] for i in order) for row in relation]),
                                     "prefix": [(q, renaming[name]) for q, name in problem.prefix]}))
    return hashlib.sha256(min(forms).encode()).hexdigest()

def transform_go(state, symmetry):
    size = state.size
    def transform(board):
        result = [0] * (size * size)
        for index, value in enumerate(board):
            r, c = divmod(index, size)
            if symmetry >= 4: c = size - 1 - c
            for _ in range(symmetry % 4): r, c = c, size - 1 - r
            result[r * size + c] = value
        return tuple(result)
    return GoState(size=size, board=transform(state.board), to_play=state.to_play, komi=state.komi,
                   history=frozenset(transform(board) for board in state.history),
                   consecutive_passes=state.consecutive_passes, move_number=state.move_number)

def canonical_go(state):
    forms = []
    for symmetry in range(8):
        value = transform_go(state, symmetry).to_dict()
        # Move count is metadata, not a rule or objective input; omit it from grouping.
        value.pop("move_number")
        forms.append(canonical_json(value))
    return hashlib.sha256(min(forms).encode()).hexdigest()

def make_csp(family, index, seed):
    rng = candidate_rng(seed, family, index)
    size = 5 if family == "graph_coloring" else 4
    colors = 3 if family == "graph_coloring" else 2
    names = [f"x{i}" for i in range(size)]
    domains = {name: tuple(range(colors)) for name in names}
    constraints = []
    if family == "graph_coloring":
        probability = rng.uniform(0.15, 0.9)
        for a, b in combinations(names, 2):
            if rng.random() < probability:
                constraints.append(TableConstraint((a, b), tuple((u, v) for u in range(colors) for v in range(colors) if u != v)))
    else:
        for _ in range(rng.randint(2, 9)):
            scope = tuple(rng.sample(names, rng.choice((1, 2, 3))))
            forbidden = tuple(rng.randrange(2) for _ in scope)
            constraints.append(TableConstraint(scope, tuple(values for values in product((0, 1), repeat=len(scope)) if values != forbidden)))
    prefix = tuple(("exists" if i % 2 == 0 else "forall", name) for i, name in enumerate(names)) if family == "quantified_sat" else ()
    return CSP(domains, tuple(constraints), prefix)

def make_go(family, index, config):
    size = int(family.split("_")[1].split("x")[0])
    rng = candidate_rng(config["seed"], family, index)
    state = initial_state(size, config["nine_board_komi"] if size == 9 else config["small_board_komi"])
    moves = []
    for _ in range(rng.randint(3, min(14, size * size))):
        placements = [action for action in legal_actions(state) if action != state.pass_action]
        if not placements: break
        action = rng.choice(placements)
        state = play(state, action)
        moves.append(action)
    player = state.to_play if index % 2 == 0 else -state.to_play
    kind = "terminal_win" if index % 7 == 0 else "area_at_horizon"
    threshold = 0.0 if kind == "terminal_win" else player * area_score(state) + rng.choice((-1.0, 0.0, 1.0))
    objective = GoObjective(kind, player, threshold)
    return state, config["go_horizons"][str(size)], objective, moves

def verify_problem(family, payload, config):
    budget = config["max_nodes_per_problem"]
    if family.startswith("go_"):
        state = GoState.from_dict(payload["state"])
        objective = GoObjective(**payload["objective"])
        result = solve_go_bounded(state, payload["horizon"], objective, max_nodes=budget)
        reference = reference_go_bounded(state, payload["horizon"], objective, max_nodes=budget)
        if result.status == SAT:
            assert verify_go_strategy(state, payload["horizon"], objective, result.strategy, max_nodes=budget).status == SAT
    else:
        problem = csp_from_dict(payload)
        if family == "quantified_sat":
            result, reference = solve_qcsp(problem, max_nodes=budget), reference_qcsp(problem, max_assignments=budget)
            if result.status == SAT: assert verify_qcsp_strategy(problem, result.strategy, max_nodes=budget).status == SAT
        else:
            result, reference = solve_csp(problem, max_nodes=budget), reference_csp(problem, max_assignments=budget)
            if result.status == SAT: assert verify_assignment(problem, result.witness)
    assert result.status == reference.status, (family, result.status, reference.status)
    return result

def candidate_rows(family, index, config):
    if family.startswith("go_"):
        state, horizon, objective, moves = make_go(family, index, config)
        group = "go-" + canonical_go(state)
        variants = [(state, 0), (transform_go(state, 5), 5)]
        payloads = [{"state": position.to_dict(), "horizon": horizon, "objective": asdict(objective)} for position, _ in variants]
        origin = {"legal_replay_actions": moves, "original_state_sha256": digest_json(state.to_dict()), "transform_symmetries": [0, 5]}
    else:
        problem = make_csp(family, index, config["seed"])
        group = "csp-" + canonical_csp(problem)
        variants = [problem, rename_csp(problem)]
        assert canonical_csp(variants[0]) == canonical_csp(variants[1])
        payloads = [csp_to_dict(value) for value in variants]
        origin = {"generator": "finite_graph_edges" if family == "graph_coloring" else "finite_boolean_clauses"}
    rows, seen = [], set()
    for variant, payload in enumerate(payloads):
        row_id = digest_json({"family": family, "problem": payload})
        if row_id in seen: continue
        seen.add(row_id)
        result = verify_problem(family, payload, config)
        rules = None
        if family.startswith("go_"):
            rules = {"profile": RULESET["name"], "board_size": payload["state"]["size"], "komi": payload["state"]["komi"],
                     "ko": "positional_superko", "passes_exempt_from_superko": True, "suicide": False,
                     "termination": "two_consecutive_passes", "scoring": "area", "dead_stones": "must_be_captured"}
        rows.append({"schema": "ahr_go_training_example_v1", "row_id": row_id, "family": family, "group_id": group,
                     "problem": payload, "rules": rules, "status": result.status, "verdict": result.status, "training_target": None if result.status == UNKNOWN else int(result.status == SAT),
                     "witness": result.witness, "strategy": result.strategy, "scope": result.scope, "reason": result.reason,
                     "provenance": {"master_seed": config["seed"], "candidate_index": index, "variant": variant,
                                    "origin": origin, "generator_code_sha256": config["generator_code_sha256"], "exports": config["export_sha256"]}})
    assert len({row["verdict"] for row in rows}) == 1
    return group, rows

## 2. Checkpointed generation, immutable files and a small resume smoke
Generation uses notebook 00's hash-checked, atomic checkpoint writer. The state
records accepted groups and the exact next candidate. Model/optimizer/scheduler/
scaler are explicitly null because this is data generation, not training. Checkpoints
are written every 8 candidates or 60 seconds and at family boundaries; latest two
and best are retained. A per-run lock rejects duplicate generation controllers.

The smoke interrupts after a small prefix, reloads serialized state, and compares
continuation byte-for-byte. The completed notebook is also rerun in a fresh kernel
as part of delivery. Larger-scale generation requires a new recovery/throughput
qualification; defaults are capped and each invocation stops safely after 8 minutes.

In [ ]:
def initial_generation_state(config):
    return {"model": None, "optimizer": None, "scheduler": None, "scaler": None,
            "rng": {"scheme": "independent_seed_per_family_and_candidate", "master_seed": config["seed"]},
            "sampler": {"family_index": 0, "next_candidate": 0}, "global_step": 0, "config": copy.deepcopy(config),
            "metrics": {"candidates": 0, "accepted_groups": 0, "optimizer_steps": 0}, "groups": {}, "complete": False}

def commit_generation(run_dir, state):
    state["global_step"] += 1
    save_checkpoint(run_dir, state, best=True, resume_cell="03: Generate or resume — restart kernel and Run All")

def run_generation(run_dir, config):
    run_dir = Path(run_dir)
    start = time.monotonic()
    with checkpoint_lock(run_dir / "controller"):
        state = load_latest_checkpoint(run_dir)
        resumed = state is not None
        if state is None: state = initial_generation_state(config)
        assert state["config"] == config, "Checkpoint configuration mismatch; use the new version directory."
        if state["complete"]: return state, {"resumed": True, "new_candidates": 0, "complete": True}
        original_candidates = state["metrics"]["candidates"]
        last_save_candidates, last_save_time = original_candidates, start
        families = list(config["groups"])
        while state["sampler"]["family_index"] < len(families):
            family = families[state["sampler"]["family_index"]]
            current = state["groups"].setdefault(family, {})
            while len(current) < config["groups"][family]:
                index = state["sampler"]["next_candidate"]
                if index >= config["max_candidates_per_family"]:
                    commit_generation(run_dir, state)
                    raise RuntimeError(f"Insufficient distinct groups for {family}; state saved. Do not silently relax splits.")
                group, rows = candidate_rows(family, index, config)
                state["sampler"]["next_candidate"] += 1
                state["metrics"]["candidates"] += 1
                if group not in current:
                    current[group] = rows
                    state["metrics"]["accepted_groups"] += 1
                elapsed = time.monotonic() - last_save_time
                if state["metrics"]["candidates"] - last_save_candidates >= config["checkpoint_candidates"] or elapsed >= config["checkpoint_seconds"]:
                    commit_generation(run_dir, state)
                    last_save_candidates, last_save_time = state["metrics"]["candidates"], time.monotonic()
                if time.monotonic() - start >= config["session_seconds"]:
                    commit_generation(run_dir, state)
                    return state, {"resumed": resumed, "new_candidates": state["metrics"]["candidates"] - original_candidates, "complete": False}
            state["sampler"] = {"family_index": state["sampler"]["family_index"] + 1, "next_candidate": 0}
            commit_generation(run_dir, state)
        state["complete"] = True
        commit_generation(run_dir, state)
        return state, {"resumed": resumed, "new_candidates": state["metrics"]["candidates"] - original_candidates, "complete": True}

def immutable_jsonl(path, rows):
    payload = "".join(canonical_json(row) + "\n" for row in rows).encode()
    if path.exists():
        assert path.read_bytes() == payload, f"Refusing to overwrite a different shard: {path}"
    else: atomic_write_bytes(path, payload)
    return {"file": path.name, "rows": len(rows), "bytes": len(payload), "sha256": hashlib.sha256(payload).hexdigest()}

def dataset_resume_smoke(project, config):
    import tempfile
    family = "boolean_sat"
    with tempfile.TemporaryDirectory(prefix="dataset-resume-", dir=Path(project) / "reports") as temporary:
        directory = Path(temporary)
        state = initial_generation_state(config)
        state["groups"][family] = {}
        for index in range(3):
            group, rows = candidate_rows(family, index, config)
            state["groups"][family][group] = rows
        state["sampler"]["next_candidate"] = 3
        state["metrics"]["candidates"] = 3
        commit_generation(directory, state)
        recovered = load_latest_checkpoint(directory)
        assert canonical_json(recovered) == canonical_json(state)
        continued = [candidate_rows(family, index, recovered["config"]) for index in range(recovered["sampler"]["next_candidate"], 6)]
        uninterrupted = [candidate_rows(family, index, config) for index in range(3, 6)]
        assert canonical_json(continued) == canonical_json(uninterrupted)
        # The refusal check operates only on temporary smoke artifacts.
        shard = directory / "test.jsonl"
        immutable_jsonl(shard, [{"value": 1}])
        try: immutable_jsonl(shard, [{"value": 2}])
        except AssertionError: pass
        else: raise AssertionError("Different immutable content was overwritten")
    return {"status": "PASS", "restored_next_candidate": 3, "next_three_candidates_identical": True,
            "overwrite_refused": True, "scope": "serialized cursor/config recovery; no model or optimizer"}

In [ ]:
smoke = dataset_resume_smoke(PROJECT_ROOT, CONFIG)
assert sum(CONFIG["groups"].values()) <= 256, "Scaling requires a separately qualified configuration."
generation_start = time.monotonic()
generation_state, generation_execution = run_generation(RUN_DIR, CONFIG)
print("Recovery smoke: PASS")
print(json.dumps(generation_execution, indent=2))
if not generation_state["complete"]:
    raise RuntimeError("Safe time limit reached and checkpoint saved. Restart kernel and Run All to continue; no final shards published yet.")
print(f"Accepted {generation_state['metrics']['accepted_groups']} groups from {generation_state['metrics']['candidates']} deterministic candidates.")

## 3. Split groups before writing train, development and final-test shards
The final test is an **experimental holdout, not a security-sealed lockbox**: files
remain accessible to the owner. This notebook validates its structure and labels
but does not print individual held-out examples or label distributions. Do not use
test shards for model fitting, selecting checkpoints, prompts, search settings or
curriculum tuning. Later training should read explicit train/dev paths only.

UNKNOWN examples go to separate **per-split** quarantine shards (`*_train_unknown`,
`*_dev_unknown`, `*_test_unknown`), retaining their assigned split
and null target. They are never silently converted into failed examples. Known
groups are split without using their SAT/UNSAT outcome; at least one known group
is reserved for development and final testing in each family. UNKNOWN groups
receive a separate deterministic hash-based split assignment.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
all_rows, shards, group_assignments, visible_counts = [], [], {}, []
for family, groups in generation_state["groups"].items():
    ordered = sorted(groups, key=lambda group: digest_json([CONFIG["seed"], group]))
    known_groups = [group for group in ordered if groups[group][0]["verdict"] != UNKNOWN]
    assert len(known_groups) >= 3, f"Need three independently verified known groups for {family}"
    test_count = max(1, len(known_groups) // 10)
    dev_count = max(1, len(known_groups) // 10)
    assignments = {group: ("test" if i < test_count else "dev" if i < test_count + dev_count else "train") for i, group in enumerate(known_groups)}
    for group in ordered:
        if group not in assignments:
            bucket = int(digest_json([CONFIG["seed"], group])[:8], 16) % 100
            assignments[group] = "test" if bucket < 10 else "dev" if bucket < 20 else "train"
    known = {split: [] for split in ("train", "dev", "test")}
    unknown = {split: [] for split in ("train", "dev", "test")}
    for group in ordered:
        split = assignments[group]
        if group in group_assignments: assert group_assignments[group] == split, "Cross-family canonical group split conflict"
        group_assignments[group] = split
        for original in groups[group]:
            row = {**original, "split": split, "dataset_version": VERSION}
            if row["verdict"] == UNKNOWN:
                assert row["training_target"] is None
                unknown[split].append(row)
            else:
                assert row["training_target"] == int(row["verdict"] == SAT)
                known[split].append(row)
            all_rows.append(row)
    for split, rows in known.items(): shards.append(immutable_jsonl(DATA_DIR / f"{family}_{split}.jsonl", rows))
    assert all(known[split] for split in known), f"Empty known split in {family}"
    for split, rows in unknown.items(): shards.append(immutable_jsonl(DATA_DIR / f"{family}_{split}_unknown.jsonl", rows))
    visible_counts.append({"family": family, "groups": len(groups), "train": len(known["train"]), "dev": len(known["dev"]), "test": len(known["test"]), "unknown": sum(map(len, unknown.values()))})

assert len({row["row_id"] for row in all_rows}) == len(all_rows), "Exact duplicate row leaked into dataset"
group_sets = {split: {row["group_id"] for row in all_rows if row["split"] == split} for split in ("train", "dev", "test")}
assert all(not group_sets[a] & group_sets[b] for a, b in combinations(group_sets, 2))
# Reload every persisted row, independently check its label, and recompute canonical grouping.
persisted_rows = []
for shard in shards:
    path = DATA_DIR / shard["file"]
    assert file_sha256(path) == shard["sha256"]
    rows = [json.loads(line) for line in path.read_text().splitlines()]
    assert len(rows) == shard["rows"]
    for row in rows:
        assert ("_" + row["split"] + ".jsonl" in shard["file"] or "_" + row["split"] + "_unknown.jsonl" in shard["file"]), "Mixed split shard"
        result = verify_problem(row["family"], row["problem"], CONFIG)
        assert result.status == row["verdict"] == row["status"]
        if row["family"].startswith("go_"):
            key = "go-" + canonical_go(GoState.from_dict(row["problem"]["state"]))
        else:
            key = "csp-" + canonical_csp(csp_from_dict(row["problem"]))
        assert key == row["group_id"]
    persisted_rows.extend(rows)
assert {row["row_id"] for row in persisted_rows} == {row["row_id"] for row in all_rows}
assert sum(shard["rows"] for shard in shards) == len(all_rows)
manifest = {"schema": "ahr_go_dataset_manifest_v1", "version": VERSION, "configuration": CONFIG,
            "rows": len(all_rows), "unique_groups": len(group_assignments), "shards": shards,
            "group_split_sha256": digest_json(group_assignments), "unknown_is_negative": False,
            "test_policy": "Held out from training and selection; not a security-sealed lockbox",
            "scope": "tiny exact CSP/QCSP and bounded Go score/terminal objectives; no eventual-game optimality claim"}
manifest_path = DATA_DIR / "manifest.json"
if manifest_path.exists(): assert json.loads(manifest_path.read_text()) == manifest, "Immutable manifest changed"
else: atomic_json(manifest_path, manifest)
print(f"{'Family':<18} {'Groups':>7} {'Train':>7} {'Dev':>7} {'Test':>7} {'Unknown':>9}")
for item in visible_counts:
    print(f"{item['family']:<18} {item['groups']:>7} {item['train']:>7} {item['dev']:>7} {item['test']:>7} {item['unknown']:>9}")
print(f"PASS: {len(shards)} shards, zero cross-split groups, all persisted labels rechecked.")

## 4. Validation receipt and takeaways
Counts below measure this starter dataset only. Generation and verification use
CPU reference algorithms intentionally; GPU training and learned AHR advantages
are not evaluated here. For training, include only `*_train.jsonl`; development
uses `*_dev.jsonl`; reserve `*_test.jsonl` for the later fixed evaluation protocol.

In [ ]:
report = {"status": "PASS", "dataset_version": VERSION, "dataset_directory": str(DATA_DIR),
          "rows": len(all_rows), "known_rows": sum(row["verdict"] != UNKNOWN for row in all_rows),
          "unknown_rows": sum(row["verdict"] == UNKNOWN for row in all_rows),
          "unique_groups": len(group_assignments), "family_count": len(visible_counts), "shard_count": len(shards),
          "by_family": visible_counts, "cross_split_group_overlap": 0, "duplicate_row_ids": 0,
          "all_persisted_labels_reverified": True, "canonical_groups_recomputed": True,
          "resume_smoke": smoke, "execution": generation_execution,
          "dataset_manifest_sha256": file_sha256(manifest_path), "runtime_seconds": round(time.monotonic() - generation_start, 3),
          "training_started": False, "claims": "Foundation only; small verified problems, no trained model or Go strength result"}
first = PROJECT_ROOT / "reports" / f"03_{VERSION}_first_pass.json"
if not first.exists(): atomic_json(first, report)
atomic_json(PROJECT_ROOT / "reports" / "03_dataset_validation.json", report)
print(json.dumps({key: report[key] for key in ("status", "rows", "known_rows", "unknown_rows", "unique_groups", "family_count", "shard_count", "runtime_seconds", "training_started")}, indent=2))